# Notebook 1: Tu primera simulación con gadget-ng

Este notebook te guía paso a paso para:
1. Compilar el binario `gadget-ng`
2. Correr una esfera de Plummer (equilibrio virial)
3. Correr una órbita kepleriana (2 cuerpos)
4. Visualizar los resultados desde Python

**Requisitos:** Rust 1.74+, Python 3.10+, matplotlib, numpy

## Paso 0: Verificar instalación de Rust

In [ ]:
import subprocess
import sys
from pathlib import Path

# Verificar que Rust esté instalado
result = subprocess.run(["rustc", "--version"], capture_output=True, text=True)
if result.returncode != 0:
    print("ERROR: Rust no encontrado.")
    print("Instalar desde: https://rustup.rs")
else:
    print(f"Rust disponible: {result.stdout.strip()}")

# Raíz del proyecto (ajustar si ejecutas desde otro directorio)
PROJECT_ROOT = Path("..")
BINARY = PROJECT_ROOT / "target" / "release" / "gadget-ng"

print(f"\nRaíz del proyecto: {PROJECT_ROOT.resolve()}")

## Paso 1: Compilar gadget-ng

La primera compilación tarda ~2-3 minutos. Las siguientes son rápidas (compilación incremental).

In [ ]:
import time

if not BINARY.exists():
    print("Compilando gadget-ng en modo release...")
    t0 = time.time()
    result = subprocess.run(
        ["cargo", "build", "--release", "-p", "gadget-ng-cli"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
    )
    elapsed = time.time() - t0
    if result.returncode != 0:
        print("ERROR de compilación:")
        print(result.stderr[-2000:])  # últimas líneas del error
    else:
        print(f"Compilado exitosamente en {elapsed:.1f}s")
else:
    print(f"Binario ya compilado: {BINARY}")

# Verificar
result = subprocess.run([str(BINARY), "--version"], capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

## Paso 2: Esfera de Plummer

Una **esfera de Plummer** es un modelo de distribución de partículas que representa
un sistema auto-gravitante en equilibrio virial (como un cúmulo globular).

El archivo de configuración `examples/plummer_sphere.toml` define:
- 512 partículas
- Radio de escala `a = 1.0 kpc`
- Solver Barnes-Hut con parámetro de apertura θ = 0.5
- 200 pasos de integración leapfrog KDK

In [ ]:
# Mostrar la configuración TOML
config_path = PROJECT_ROOT / "examples" / "plummer_sphere.toml"
print(config_path.read_text())

In [ ]:
# Correr la simulación
out_plummer = PROJECT_ROOT / "runs" / "plummer_nb01"

print("Corriendo esfera de Plummer (512 partículas, 200 pasos)...")
t0 = time.time()

result = subprocess.run(
    [
        str(BINARY), "stepping",
        "--config", str(PROJECT_ROOT / "examples" / "plummer_sphere.toml"),
        "--out", str(out_plummer),
        "--snapshot",
    ],
    capture_output=True,
    text=True,
)

elapsed = time.time() - t0
print(f"Finalizado en {elapsed:.2f}s")

if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
else:
    print("Salida:", result.stdout[-500:] if result.stdout else "(sin stdout)")
    print("\nArchivos generados:")
    for f in sorted(out_plummer.rglob("*")):
        print(f"  {f.relative_to(out_plummer)}")

## Paso 3: Leer y visualizar el snapshot final

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm


def load_snapshot_jsonl(snapshot_dir: Path) -> dict:
    """Carga un snapshot en formato JSONL y devuelve arrays numpy."""
    particles_file = snapshot_dir / "particles.jsonl"
    if not particles_file.exists():
        # Puede estar bajo snapshot_final/
        alt = list(snapshot_dir.glob("*/particles.jsonl"))
        if alt:
            particles_file = alt[0]
        else:
            raise FileNotFoundError(f"No se encontró particles.jsonl en {snapshot_dir}")

    positions, velocities, masses = [], [], []
    with open(particles_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            p = json.loads(line)
            positions.append(p.get("pos", p.get("position", [0, 0, 0])))
            velocities.append(p.get("vel", p.get("velocity", [0, 0, 0])))
            masses.append(p.get("mass", 1.0))

    return {
        "pos": np.array(positions),
        "vel": np.array(velocities),
        "mass": np.array(masses),
    }


# Intentar cargar el snapshot
snap_dir = out_plummer / "snapshot_final"
if not snap_dir.exists():
    # Puede llamarse de otra forma
    candidates = list(out_plummer.glob("snapshot*"))
    snap_dir = candidates[0] if candidates else out_plummer

try:
    data = load_snapshot_jsonl(snap_dir)
    print(f"Partículas cargadas: {len(data['pos'])}")
    print(f"Rango X: [{data['pos'][:,0].min():.3f}, {data['pos'][:,0].max():.3f}]")
    print(f"Rango Y: [{data['pos'][:,1].min():.3f}, {data['pos'][:,1].max():.3f}]")
except FileNotFoundError as e:
    print(f"No se pudo cargar el snapshot: {e}")
    print("Archivos disponibles:")
    for f in out_plummer.rglob("*"):
        print(f"  {f}")
    data = None

In [ ]:
if data is not None:
    pos = data["pos"]
    vel = data["vel"]
    speed = np.linalg.norm(vel, axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("Esfera de Plummer — estado final", fontsize=14)

    # Proyección XY coloreada por velocidad
    sc = axes[0].scatter(pos[:, 0], pos[:, 1], c=speed, cmap="plasma", s=3, alpha=0.8)
    axes[0].set_xlabel("X [kpc]")
    axes[0].set_ylabel("Y [kpc]")
    axes[0].set_title("Proyección XY")
    axes[0].set_aspect("equal")
    plt.colorbar(sc, ax=axes[0], label="|v| [km/s]")

    # Proyección XZ
    axes[1].scatter(pos[:, 0], pos[:, 2], c=speed, cmap="plasma", s=3, alpha=0.8)
    axes[1].set_xlabel("X [kpc]")
    axes[1].set_ylabel("Z [kpc]")
    axes[1].set_title("Proyección XZ")
    axes[1].set_aspect("equal")

    # Perfil de densidad radial
    center = pos.mean(axis=0)
    r = np.linalg.norm(pos - center, axis=1)
    r_bins = np.linspace(0, r.max(), 30)
    counts, edges = np.histogram(r, bins=r_bins)
    r_centers = 0.5 * (edges[:-1] + edges[1:])
    vol = (4 / 3) * np.pi * (edges[1:] ** 3 - edges[:-1] ** 3)
    density = counts / vol

    # Perfil de Plummer teórico: ρ(r) ∝ (1 + r²/a²)^{-5/2}
    a = 1.0  # radio de escala en kpc
    r_theory = np.linspace(0.05, r.max(), 200)
    rho_theory = (1 + (r_theory / a) ** 2) ** (-2.5)
    rho_theory *= density[density > 0][0] / rho_theory[0]  # normalizar

    axes[2].plot(r_centers[density > 0], density[density > 0], "o", label="Simulación", ms=4)
    axes[2].plot(r_theory, rho_theory, "--", label="Plummer teórico", lw=2)
    axes[2].set_xlabel("r [kpc]")
    axes[2].set_ylabel("Densidad numérica")
    axes[2].set_title("Perfil radial de densidad")
    axes[2].set_yscale("log")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Paso 4: Órbita Kepleriana

Este ejemplo simula el sistema Sol-Tierra con gravedad directa N².
Verifica la conservación de energía y el período orbital.

In [ ]:
out_kepler = PROJECT_ROOT / "runs" / "kepler_nb01"

print("Corriendo órbita kepleriana (2 partículas, 1000 pasos)...")
t0 = time.time()

result = subprocess.run(
    [
        str(BINARY), "stepping",
        "--config", str(PROJECT_ROOT / "examples" / "kepler_orbit.toml"),
        "--out", str(out_kepler),
        "--snapshot",
    ],
    capture_output=True,
    text=True,
)

elapsed = time.time() - t0
print(f"Finalizado en {elapsed:.3f}s")

if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])
else:
    print("OK")

## Resumen

En este notebook aprendiste a:
- Compilar `gadget-ng` desde Python
- Ejecutar simulaciones con `stepping`
- Leer snapshots JSONL
- Visualizar proyecciones y comparar con perfiles teóricos

**Siguientes notebooks:**
- `02_simulacion_cosmologica.ipynb` — ΛCDM con espectro de potencia P(k)
- `03_herramientas_analisis.ipynb` — FoF, P(k), función de correlación ξ(r)